In [1]:
import cv2
import numpy as np
import os


In [10]:
path_base = 'D:\\Teach_n_Train\\Advanced_Lessons_CV\\LABS\\Yolo\\yolo_opencv\\'

input_image  = 'traffic.jpg'
file_classes = path_base + 'yolov3.txt'
file_weights = path_base + 'yolov3.weights'
file_config  = path_base + 'yolov3.cfg'


In [11]:
def get_output_layers(yolo_model):
    
    layer_names = yolo_model.getLayerNames()
        
    output_layers = [layer_names[int(i[0]) - 1] for i in 
                            yolo_model.getUnconnectedOutLayers()]

    return output_layers


In [12]:
def draw_prediction(img, class_id, confidence, x, y, x_plus_w, y_plus_h):

    label = str(classes[class_id])
    color = COLORS[class_id]

    cv2.rectangle(img, (x,y), (x_plus_w,y_plus_h), color, 2)
    cv2.putText(img, label, (x-10,y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)


In [13]:
# load image by cv2
image = cv2.imread(path_base + input_image)
#keep file name to save result image
filename = os.path.splitext(input_image)[0]  

Width = image.shape[1]
Height = image.shape[0]
scale = 0.00392

# load class name (name of object categores)
classes = None

with open(file_classes, 'r') as f:
    classes = [line.strip() for line in f.readlines()]

# set of color to draw boxes w.r.t class of object
COLORS = np.random.uniform(0, 255, size=(len(classes), 3))


In [14]:
# Load Model Yolo w.r.t config and weights ready
yolo_model = cv2.dnn.readNet(file_weights, file_config)

# set input: resize to 416x416
blob = cv2.dnn.blobFromImage(image, scale, (416,416), (0,0,0), True, crop=False)

yolo_model.setInput(blob)

# perform predict w.r.t model which loaded
outs = yolo_model.forward(get_output_layers(yolo_model))


In [15]:
class_ids = []
confidences = []
boxes = []
conf_threshold = 0.5
nms_threshold = 0.4
for out in outs:
    for detection in out:
        scores = detection[5:]
        class_id = np.argmax(scores)
        confidence = scores[class_id]
        if confidence > 0.5:
            center_x = int(detection[0] * Width)
            center_y = int(detection[1] * Height)
            w = int(detection[2] * Width)
            h = int(detection[3] * Height)
            x = center_x - w / 2
            y = center_y - h / 2
            class_ids.append(class_id)
            confidences.append(float(confidence))
            boxes.append([x, y, w, h])

indices = cv2.dnn.NMSBoxes(boxes, confidences, conf_threshold, nms_threshold)


In [16]:
for i in indices:
    i = i[0]
    box = boxes[i]
    x = box[0]
    y = box[1]
    w = box[2]
    h = box[3]
    draw_prediction(image, class_ids[i], confidences[i], round(x), round(y), round(x+w), round(y+h))

cv2.imshow("object detection", image)
cv2.waitKey()

cv2.imwrite(filename + "-object-detection.jpg", image)
cv2.destroyAllWindows()
